# LLM Fine-Tuning Benchmark — Bengali Yellow Journalism Detection

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Dataset:** Swarabyanjan Clean Balanced Dataset (766 samples: 383 yellow + 383 non-yellow)
**Environment:** Kaggle T4 x2, Internet enabled, commit mode

---

## Model Loading Strategy

This notebook supports two model loading paths, with automatic fallback:

1. **Kaggle Models (preferred):** Models are added as Kaggle Inputs via the "Add Models" feature. Loading is instantaneous and requires no HuggingFace download.
2. **HuggingFace Hub (fallback):** If a model is not available as a Kaggle Input, the notebook falls back to downloading from HuggingFace Hub using `hf_transfer` for accelerated transfer (2-3x speedup).

## Required Kaggle Inputs

### 1. Gold Dataset (mandatory)
- **Search:** `swarabyanjan` or `v18-human-gold-final`
- **File:** `Swarabyanjan_BEST_BALANCED_1to1.csv`

### 2. Kaggle Models (add as many as available)
Navigate to **Add Input > Models** and add the following models. If a model is not found on Kaggle, the notebook will automatically fall back to HuggingFace download.

The table below lists the full model registry from the benchmark. This notebook runs only **Qwen2.5-3B-Instruct** (highlighted below).

| # | Model | Kaggle Status | Verified Path |
|---|-------|---------------|---------------|
| 1 | Gemma-2-2B-it | ✅ Available | `/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2` |
| 2 | Qwen2.5-3B-Instruct | ✅ Available | `/kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1` |
| 3 | Llama-3.2-3B-Instruct | ❌ Not added | HuggingFace fallback |
| 4 | Phi-3-mini-4k | ✅ Available | `/kaggle/input/models/Microsoft/phi-3/pytorch/phi-3.5-mini-instruct/2` |
| 5 | Mistral-7B-v0.2 | ❌ Not on Kaggle | HuggingFace fallback |
| 6 | Mistral-7B-v0.3 | ❌ Not on Kaggle | HuggingFace fallback |
| 7 | Zephyr-7B-beta | ❌ Not on Kaggle | HuggingFace fallback |
| 8 | Qwen2.5-7B-Instruct | ✅ Available | `/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1` |
| 9 | Llama-3.1-8B-Instruct | ✅ Available | `/kaggle/input/models/metaresearch/llama-3.1/transformers/8b/2` |
| 10 | Gemma-2-9B-it | ✅ Available | `/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b/2` |

**Summary (verified 2026-07-14):**
- 6 models available on Kaggle (instant load, no download)
- 4 models will use HuggingFace fallback (Mistral v0.2, v0.3, Zephyr, Llama-3.2-3B)
- Dataset verified at `/kaggle/input/datasets/swagotammalakar/v18-human-gold-final/`
- HF_TOKEN required for the 4 fallback models (Mistral and Zephyr are open-access; Llama-3.2-3B is gated)

### 3. HuggingFace Token (for fallback / gated models)
- **Secret name:** `HF_TOKEN`
- **Required only if:** a model is not available on Kaggle Models and must be downloaded from HuggingFace
- Gated models (Gemma, Llama, Phi) require license acceptance on their HuggingFace model page before the token works

## Execution Architecture

| Feature | Implementation |
|---------|----------------|
| **Parallelism** | Sequential (QLoRA 4-bit quantisation is incompatible with `DataParallel`; bitsandbytes parameters are pinned to a single device) |
| **GPU** | Single T4 (`CUDA_VISIBLE_DEVICES=0`); second T4 remains idle for stability |
| **Precision** | `float16` (T4 Turing architecture does not support `bfloat16`) |
| **Logging** | Per-model cells with `flush=True` and `sys.stdout.flush()` — logs appear immediately after each cell completes in Kaggle commit mode |
| **Disk management** | Model files are deleted after training and evaluation to free disk space for subsequent models |
| **Resumability** | Completed models are persisted to a results CSV after each evaluation; interrupted runs resume from the last completed model |
| **Error isolation** | Each model is wrapped in an independent `try/except` block; a single failure does not terminate the benchmark |

## Expected Runtime

| Configuration | Duration |
|---------------|----------|
| All models on Kaggle (no download) | ~5-6 hours |
| Mixed (some Kaggle, some HuggingFace) | ~6-8 hours |
| All models from HuggingFace | ~8-10 hours |
| Kaggle 12-hour session limit | Fits comfortably |

## Kaggle Setup

| Setting | Value |
|---------|-------|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** |
| Expected runtime | ~81 minutes |

**Required Kaggle Inputs:**
- Dataset: `swagotammalakar/swarabyanjan` (provides `Swarabyanjan_Gold_Balanced_766.csv`)
- Model: `qwen-lm/qwen2.5/transformers/3b-instruct`

> QLoRA 4-bit fine-tuning.
>
> **Note:** If Kaggle resets the inputs/accelerator after re-importing the notebook, re-attach the dataset + model listed above and set accelerator to GPU T4 ×2 before running.


### 1. Environment Setup and Library Imports

The Kaggle T4 environment ships with PyTorch, transformers, and scikit-learn pre-installed. The `bitsandbytes`, `trl`, `peft`, `accelerate`, and `hf_transfer` packages are installed at the start of the run. `hf_transfer` provides 2-3x download acceleration when the HuggingFace fallback path is used.

In [1]:
%%capture _install
!pip install -q bitsandbytes trl peft accelerate datasets hf_transfer

# --- GPU isolation ---
# QLoRA 4-bit models (bitsandbytes) cannot use DataParallel — their parameters
# are pinned to a single device. Setting CUDA_VISIBLE_DEVICES=0 makes only
# one GPU visible to PyTorch, preventing DataParallel errors.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # 2-3x faster HuggingFace download

import torch
import numpy as np
import pandas as pd
import gc, time, warnings, json, sys, re, glob, shutil
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, Tuple
import inspect

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, set_seed
)
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

warnings.filterwarnings('ignore')
set_seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

def _now():
    """Timestamp helper for logging."""
    return datetime.now().strftime('%H:%M:%S')

print(f"[{_now()}] PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"[{_now()}] GPUs visible: {n_gpus}", flush=True)
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} (compute {props.major}.{props.minor})", flush=True)
else:
    print(f"[{_now()}] WARNING: No GPU detected.", flush=True)

### 2. GPU Detection and Disk Space Audit

The T4 GPU (Turing architecture, compute capability 7.5) does not support `bfloat16` operations natively; `float16` is used throughout. Disk space is audited before training begins, as 4-bit quantised models consume 2-6 GB each and the HuggingFace cache can grow rapidly.

In [2]:
# GPU and disk audit
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader
    print(f"\ntorch.cuda.device_count() = {torch.cuda.device_count()}", flush=True)

# Disk space check
disk = os.statvfs('/kaggle/working')
free_gb = (disk.f_bavail * disk.f_frsize) / 1e9
total_gb = (disk.f_blocks * disk.f_frsize) / 1e9
print(f"[{_now()}] Disk space: {free_gb:.1f} GB free / {total_gb:.1f} GB total", flush=True)
print(f"[{_now()}] Working directory: /kaggle/working", flush=True)

0, Tesla T4, 15360 MiB, 14909 MiB
1, Tesla T4, 15360 MiB, 14912 MiB

torch.cuda.device_count() = 1
[10:41:29] Disk space: 20.9 GB free / 21.0 GB total
[10:41:29] Working directory: /kaggle/working


### 3. Configuration — Model Registry and Hyperparameters

Each model entry specifies its HuggingFace ID, display name, and a list of candidate Kaggle Input paths. The notebook searches these paths in order; the first match is used. If no Kaggle path is found, the HuggingFace Hub is used as a fallback. Models are ordered smallest-first to provide early feedback on pipeline correctness.

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
MAX_SEQ_LEN   = 512
NUM_EPOCHS    = 3  # More epochs for smaller dataset (766 samples)
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO  = 0.03
LORA_R        = 16
LORA_ALPHA    = 32
TRAIN_FRAC    = 0.8   # 612 train / 154 test (766 total)

GOLD_CSV_FILENAME = 'Swarabyanjan_BEST_BALANCED_1to1.csv'
RESULTS_FILE  = '/kaggle/working/local_llm_results.csv'
OUTPUT_DIR    = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a Bengali news analyst. Classify the given news article as "
    "yellow journalism (1) or not (0). Answer with only the digit 1 or 0."
)

# --- Model registry ---
# Each entry: (display_name, hf_id, kaggle_input_paths)
# kaggle_input_paths: list of candidate paths to check (glob patterns).
# The first match is used. If none match, HuggingFace Hub is used.
MODEL_CONFIGS = [
    {
        "name": "Qwen2.5-3B-Instruct",
        "hf_id": "Qwen/Qwen2.5-3B-Instruct",
        "kaggle_paths": ["/kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/*"],
    },
]

set_seed(SEED)

def find_model_path(cfg):
    """Find local model path. Returns (path, source) tuple.
    source: 'kaggle' or 'huggingface'.
    """
    for pattern in cfg.get("kaggle_paths", []):
        matches = sorted(glob.glob(pattern))
        if matches:
            # Verify it has model files
            model_path = matches[0]
            has_safetensors = any(
                f.endswith('.safetensors') or f.endswith('.bin')
                for dp, _, fns in os.walk(model_path)
                for f in fns
            )
            if has_safetensors:
                return model_path, "kaggle"
    return cfg["hf_id"], "huggingface"

print(f"[{_now()}] Models configured: {len(MODEL_CONFIGS)}", flush=True)
print(f"[{_now()}] Hyperparameters: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, "
      f"grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}, max_len={MAX_SEQ_LEN}", flush=True)
print(f"[{_now()}] Checking model availability...", flush=True)
for cfg in MODEL_CONFIGS:
    path, source = find_model_path(cfg)
    icon = "KAGGLE" if source == "kaggle" else "HF-HUB"
    print(f"  [{icon}] {cfg['name']:<28} -> {path}", flush=True)

[10:41:29] Models configured: 1
[10:41:29] Hyperparameters: epochs=3, batch=2, grad_accum=4, lr=0.0002, max_len=512
[10:41:29] Checking model availability...
  [KAGGLE] Qwen2.5-3B-Instruct          -> /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1


### 5. Dataset Loading with Path Auto-Discovery

The clean balanced dataset CSV is located by searching a prioritised list of candidate Kaggle mount paths, followed by a recursive glob search under `/kaggle/input/`. The headline and body preview columns are concatenated into a single text field, as this combination provides the full article context required for yellow journalism classification.

In [4]:
def find_gold_csv():
    candidates = [
        '/kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_BEST_BALANCED_1to1.csv',
        '/kaggle/input/swarabyanjan/Swarabyanjan_BEST_BALANCED_1to1.csv',
        f'/kaggle/input/datasets/swagotammalakar/v18-human-gold-final/{GOLD_CSV_FILENAME}',  # VERIFIED
        f'/kaggle/input/swarabyanjan/{GOLD_CSV_FILENAME}',
        f'/kaggle/input/v18-human-gold-final/{GOLD_CSV_FILENAME}',
    ]
    for c in candidates:
        if os.path.isfile(c): return c
    matches = glob.glob(f'/kaggle/input/**/{GOLD_CSV_FILENAME}', recursive=True)
    if matches: return matches[0]
    matches = glob.glob('/kaggle/input/**/Swarabyanjan_BEST_BALANCED*.csv', recursive=True)
    if matches: return matches[0]
    return candidates[0]

DATA_PATH = find_gold_csv()
print(f"[{_now()}] Dataset: {DATA_PATH}", flush=True)
print(f"[{_now()}] Exists: {os.path.isfile(DATA_PATH)}", flush=True)

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['headline', 'body_text', 'best_label']).reset_index(drop=True)
df['best_label'] = df['best_label'].astype(int)
df['article'] = df['headline'].fillna('').astype(str) + '\n\n' + df['body_text'].fillna('').astype(str)
TEXT_COL = 'article'
LABEL_COL = 'best_label'

print(f"[{_now()}] Loaded: {df.shape}", flush=True)
print(f"Label distribution:\n{df[LABEL_COL].value_counts().sort_index().to_string()}", flush=True)

[10:41:29] Dataset: /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_BEST_BALANCED_1to1.csv
[10:41:29] Exists: True
[10:41:29] Loaded: (766, 9)
Label distribution:
best_label
0    383
1    383


### 6. Stratified Train-Test Split

An 80/20 stratified split (612 training / 154 test) is used. Stratification preserves the 50:50 class balance in both partitions. The split is fixed by `random_state=42` to ensure reproducibility.

In [5]:
train_df, test_df = train_test_split(
    df[[TEXT_COL, LABEL_COL]], test_size=1-TRAIN_FRAC,
    stratify=df[LABEL_COL], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"[{_now()}] Train: {len(train_df)} | Test: {len(test_df)}", flush=True)
print(f"Train: {train_df[LABEL_COL].value_counts().sort_index().to_dict()}", flush=True)
print(f"Test:  {test_df[LABEL_COL].value_counts().sort_index().to_dict()}", flush=True)

[10:41:29] Train: 612 | Test: 154
Train: {0: 306, 1: 306}
Test:  {0: 77, 1: 77}


### 7. Utility Functions — Metrics, Parsing, Disk Management

The `parse_prediction` function extracts a binary label from the model's generated text using a three-stage fallback: first-character check, full-text digit scan, and Bengali numeral matching. Unparseable predictions are tracked separately and excluded from the primary metric computation. The `cleanup_model_files` function deletes the model directory after training to free disk space for subsequent models.

In [6]:
def compute_metrics(y_true, y_pred, model_name, n_unparseable=0):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    parseable_pct = 100.0 * (len(y_true) - n_unparseable) / max(len(y_true), 1)
    print(f"\n{'='*55}", flush=True)
    print(f"  {model_name}", flush=True)
    print(f"{'='*55}", flush=True)
    print(f"  Acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}", flush=True)
    print(f"  Kappa={kappa:.4f}  MCC={mcc:.4f}", flush=True)
    print(f"  TP={tp} FP={fp} FN={fn} TN={tn}", flush=True)
    print(f"  Parseable: {len(y_true)-n_unparseable}/{len(y_true)} ({parseable_pct:.1f}%)", flush=True)
    print(f"{'='*55}", flush=True)
    return {"Model": model_name, "Accuracy": round(acc,4), "Precision": round(prec,4),
            "Recall": round(rec,4), "F1": round(f1,4), "Kappa": round(kappa,4),
            "MCC": round(mcc,4), "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
            "Unparseable": int(n_unparseable), "Parseable_Pct": round(parseable_pct,1)}

def parse_prediction(text):
    if not text or not text.strip(): return -1
    text = text.strip()
    if text[0] == "1": return 1
    if text[0] == "0": return 0
    for ch in text:
        if ch == "1": return 1
        if ch == "0": return 0
    bengali_map = {"\u09e7": 1, "\u09e6": 0}  # ১, ০
    for ch in text:
        if ch in bengali_map: return bengali_map[ch]
    return -1

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def cleanup_model_files(model_path, model_name):
    """Delete model files to free disk space. Only deletes HuggingFace
    cache, not Kaggle Input files (those are read-only)."""
    if model_path.startswith("/kaggle/input/"):
        print(f"[{_now()}] Kaggle Input model — not deleting (read-only)", flush=True)
        return
    # HuggingFace cache cleanup
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    if os.path.exists(cache_dir):
        cache_size = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fns in os.walk(cache_dir) for f in fns) / 1e9
        shutil.rmtree(cache_dir, ignore_errors=True)
        print(f"[{_now()}] HF cache cleared: {cache_size:.2f} GB freed", flush=True)
    # Local lora output
    lora_dir = f"./lora_{model_name.replace('/', '_')}"
    if os.path.exists(lora_dir):
        shutil.rmtree(lora_dir, ignore_errors=True)

def plot_cm(y_true, y_pred, model_name, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not YJ","Yellow J."], yticklabels=["Not YJ","Yellow J."])
    plt.title(model_name, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

print(f"[{_now()}] Utilities ready.", flush=True)

[10:41:29] Utilities ready.


### 8. Fine-Tuning and Evaluation Pipeline

Each model is processed through a six-stage pipeline: (1) tokenizer loading, (2) 4-bit NF4 quantised model loading, (3) chat-template-based data formatting, (4) LoRA configuration with auto-detected target modules, (5) supervised fine-tuning with gradient checkpointing, and (6) greedy-decoding evaluation on the held-out test set. The `SFTTrainer` API is version-safe: the constructor signature is inspected and only supported parameters are passed.

In [7]:
def finetune_and_evaluate(model_id, model_name, model_source="auto"):
    """Full pipeline: Load 4-bit -> Format -> LoRA -> SFT -> Eval -> Cleanup."""
    tag = model_name
    print(f"\n[{_now()}] {'='*60}", flush=True)
    print(f"[{_now()}] {tag}", flush=True)
    print(f"[{_now()}] Source: {model_id}", flush=True)
    print(f"[{_now()}] {'='*60}", flush=True)
    t0 = time.time()

    # --- 1. Tokenizer ---
    print(f"[{_now()}] [{tag}] 1/6 Loading tokenizer...", flush=True)
    tok_kwargs = {"trust_remote_code": True}
    if os.environ.get("HF_TOKEN"):
        tok_kwargs["token"] = os.environ["HF_TOKEN"]
    tokenizer = AutoTokenizer.from_pretrained(model_id, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # FIX 1: Chat template fallback — some models (Gemma-2-2B) don't have it
    if not hasattr(tokenizer, 'chat_template') or tokenizer.chat_template is None:
        print(f"[{_now()}] [{tag}] No chat_template found — using default.", flush=True)
        tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'system' %}{{ message['content'] }}\n{% elif message['role'] == 'user' %}User: {{ message['content'] }}\n{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }}\n{% endif %}{% endfor %}Assistant: "

    # --- 2. Model (4-bit, fp16) ---
    print(f"[{_now()}] [{tag}] 2/6 Loading model (4-bit NF4, fp16)...", flush=True)
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs = {
        "quantization_config": bnb_cfg,
        "device_map": {"": 0},
        "trust_remote_code": True,
        "torch_dtype": torch.float16,
    }
    if os.environ.get("HF_TOKEN"):
        model_kwargs["token"] = os.environ["HF_TOKEN"]
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    # FIX 2: Force float16 — some models load as bfloat16, causing AMP errors on T4
    model.config.torch_dtype = torch.float16
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model.enable_input_require_grads()
    print(f"[{_now()}] [{tag}] VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB", flush=True)
    sys.stdout.flush()

    # --- 3. Format data ---
    print(f"[{_now()}] [{tag}] 3/6 Formatting data...", flush=True)
    def _fmt(row):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(row[TEXT_COL])[:2000]},
            {"role": "assistant", "content": str(int(row[LABEL_COL]))},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}
    tr_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]])
    tr_ds = tr_ds.map(_fmt, remove_columns=tr_ds.column_names)

    te_prompts, te_labels = [], []
    for _, row in test_df.iterrows():
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(row[TEXT_COL])[:2000]},
        ]
        te_prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        te_labels.append(int(row[LABEL_COL]))

    # --- 4. LoRA ---
    print(f"[{_now()}] [{tag}] 4/6 Setting up LoRA...", flush=True)
    preferred = ["q_proj", "k_proj", "v_proj", "o_proj"]
    named = set(n for n, _ in model.named_modules())
    targets = [m for m in preferred if any(m in n for n in named)]
    if not targets: targets = ["q_proj", "v_proj"]
    print(f"[{_now()}] [{tag}] LoRA targets: {targets}", flush=True)

    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=targets,
                          lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
    args = TrainingArguments(
        output_dir=f"./lora_{model_name.replace('/', '_')}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        fp16=False,
        bf16=False,
        logging_steps=25,
        disable_tqdm=False,
        report_to="none",
        save_strategy="no",
        gradient_checkpointing=False,
        optim="paged_adamw_8bit",
        seed=SEED,
        dataloader_pin_memory=False,
        max_grad_norm=1.0,
    )

    # Custom callback for real-time progress logging in Kaggle commit mode
    from transformers import TrainerCallback
    class FlushProgressCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is not None:
                epoch = logs.get('epoch', 0)
                loss = logs.get('loss', 0)
                lr = logs.get('learning_rate', 0)
                step = state.global_step
                total = state.max_steps
                pct = 100 * step / total if total > 0 else 0
                print(f"[{_now()}] [{tag}] step {step}/{total} ({pct:.1f}%) | "
                      f"epoch={epoch:.2f} | loss={loss:.4f} | lr={lr:.2e}", flush=True)
    
    # Version-safe SFTTrainer
    sig = set(inspect.signature(SFTTrainer.__init__).parameters)
    tr_kwargs = {"model": model, "train_dataset": tr_ds, "args": args, "peft_config": lora_cfg}
    if "processing_class" in sig: tr_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig: tr_kwargs["tokenizer"] = tokenizer
    if "max_seq_length" in sig: tr_kwargs["max_seq_length"] = MAX_SEQ_LEN
    elif "max_seq_len" in sig: tr_kwargs["max_seq_len"] = MAX_SEQ_LEN
    if "dataset_text_field" in sig: tr_kwargs["dataset_text_field"] = "text"
    trainer = SFTTrainer(**tr_kwargs)
    trainer.add_callback(FlushProgressCallback())

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[{_now()}] [{tag}] Trainable: {trainable:,}/{total:,} ({100*trainable/total:.2f}%)", flush=True)
    sys.stdout.flush()

    # --- 5. Train ---
    print(f"[{_now()}] [{tag}] 5/6 Training {NUM_EPOCHS} epoch(s)...", flush=True)
    t_train = time.time()
    trainer.train()
    train_min = (time.time() - t_train) / 60
    print(f"[{_now()}] [{tag}] Training: {train_min:.1f} min", flush=True)
    sys.stdout.flush()

    # --- 6. Evaluate ---
    print(f"[{_now()}] [{tag}] 6/6 Evaluating {len(te_prompts)} samples...", flush=True)
    t_eval = time.time()
    preds, unparseable = [], 0
    model.eval()
    for i, prompt in enumerate(te_prompts):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=3, do_sample=False,
                                 temperature=0.0, pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        p = parse_prediction(gen)
        preds.append(p if p != -1 else 0)
        if p == -1: unparseable += 1
        if (i+1) % 100 == 0:
            el = (time.time()-t_eval)/60
            print(f"[{_now()}] [{tag}] {i+1}/{len(te_prompts)} ({el:.1f}m, unparseable={unparseable})", flush=True)
    eval_min = (time.time()-t_eval)/60

    metrics = compute_metrics(te_labels, preds, model_name, unparseable)
    metrics["Train_Min"] = round(train_min, 1)
    metrics["Eval_Min"] = round(eval_min, 1)
    metrics["Source"] = model_source
    metrics["Total_Min"] = round((time.time()-t0)/60, 1)

    safe = model_name.replace("/", "_").replace(".", "_")
    plot_cm(te_labels, preds, model_name, f"{OUTPUT_DIR}/cm_{safe}.png")
    print(f"[{_now()}] [{tag}] TOTAL: {metrics['Total_Min']:.1f} min", flush=True)

    # Cleanup — AGGRESSIVE GPU memory clearing for next model
    # This is critical: without proper cleanup, next model will OOM
    try:
        del model
    except:
        pass
    try:
        del trainer
    except:
        pass
    try:
        del tokenizer
    except:
        pass
    try:
        del tr_ds
    except:
        pass
    
    # Force garbage collection
    import gc
    gc.collect()
    gc.collect()  # twice for thoroughness
    
    # Clear CUDA cache — this releases reserved-but-unused memory
    if torch.cuda.is_available():
        torch.cuda.synchronize()    # wait for all async ops to finish
        torch.cuda.empty_cache()    # clear cache
        torch.cuda.synchronize()    # sync again
        torch.cuda.empty_cache()    # clear again (paranoid)
        
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"[{_now()}] [{tag}] GPU cleanup done | allocated: {allocated:.2f} GB | "
              f"reserved: {reserved:.2f} GB", flush=True)
        
        # If still high memory, print warning
        if allocated > 2.0:
            print(f"[{_now()}] [{tag}] WARNING: GPU still has {allocated:.2f} GB allocated "
                  f"— next model may OOM", flush=True)
    
    # Clean up disk
    cleanup_model_files(model_id, model_name)
    sys.stdout.flush()
    return metrics

print(f"[{_now()}] finetune_and_evaluate() ready.", flush=True)

[10:41:29] finetune_and_evaluate() ready.


### 9. Benchmark Execution — One Cell Per Model

Each model runs in its own cell. In Kaggle commit mode, a cell's logs appear immediately after that cell completes, providing real-time visibility into training progress. The `run_single_model` helper handles model path resolution, training, evaluation, and result persistence.

### 8.5 Input Diagnostic — Verify Kaggle Models Before Training

This cell lists all Kaggle Inputs currently attached to the notebook. Run this cell first and paste the output back to the assistant to verify that all model paths are correct before starting the benchmark.


In [8]:
# ============================================================
# DIAGNOSTIC: List all Kaggle Input models and datasets
# ============================================================
# This cell shows what Kaggle Inputs are currently attached.
# Paste the output back to me so I can verify the paths.

import os, glob

print("="*60, flush=True)
print("KAGGLE INPUTS DIAGNOSTIC", flush=True)
print("="*60, flush=True)

# List all directories under /kaggle/input/
input_base = '/kaggle/input'
if os.path.exists(input_base):
    print(f"\nAll inputs under {input_base}:", flush=True)
    for item in sorted(os.listdir(input_base)):
        full = os.path.join(input_base, item)
        if os.path.isdir(full):
            print(f"  {item}/", flush=True)
else:
    print(f"ERROR: {input_base} does not exist", flush=True)

# Find all model directories (look for config.json or model files)
print(f"\n{'='*60}", flush=True)
print("MODEL DIRECTORIES (with config.json or safetensors):", flush=True)
print(f"{'='*60}", flush=True)

model_dirs = []
for root, dirs, files in os.walk('/kaggle/input'):
    # Skip __pycache__ and hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    if 'config.json' in files:
        # Check if there are model weight files
        has_weights = any(f.endswith('.safetensors') or f.endswith('.bin') for f in files)
        if has_weights:
            model_dirs.append(root)

if model_dirs:
    for d in sorted(model_dirs):
        # Get size
        total_size = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fns in os.walk(d) for f in fns) / 1e9
        print(f"  {d}  ({total_size:.2f} GB)", flush=True)
else:
    print("  No model directories found.", flush=True)

# Also list any CSV files (dataset)
print(f"\n{'='*60}", flush=True)
print("CSV/DATASET FILES:", flush=True)
print(f"{'='*60}", flush=True)
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
for f in sorted(csv_files)[:10]:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)", flush=True)

# Summary: which of our 10 models are found?
print(f"\n{'='*60}", flush=True)
print("MODEL AVAILABILITY SUMMARY", flush=True)
print(f"{'='*60}", flush=True)

expected = {
    "Gemma-2-2B": ["gemma-2-2b", "gemma2-2b"],
    "Qwen2.5-3B": ["3b-instruct", "qwen2.5-3b"],
    "Llama-3.2-3B": ["3b-instruct", "llama-3.2-3b"],
    "Phi-3": ["phi-3", "phi-3.5-mini"],
    "Qwen2.5-7B": ["7b-instruct", "qwen2.5-7b"],
    "Llama-3.1-8B": ["8b", "llama-3.1-8b"],
    "Gemma-2-7B": ["gemma-2-7b", "gemma2-7b"],
    "Mistral": ["mistral", "7b-instruct"],
    "Zephyr": ["zephyr"],
}

all_paths_str = ' '.join(model_dirs).lower()
for name, keywords in expected.items():
    found = any(kw.lower() in all_paths_str for kw in keywords)
    icon = "FOUND" if found else "NOT FOUND"
    print(f"  [{icon}] {name}", flush=True)

print(f"\nTotal model directories: {len(model_dirs)}", flush=True)
print(f"\n>>> Paste the above output back to me <<<", flush=True)


KAGGLE INPUTS DIAGNOSTIC

All inputs under /kaggle/input:
  datasets/
  models/

MODEL DIRECTORIES (with config.json or safetensors):
  /kaggle/input/models/Microsoft/phi-3/pytorch/phi-3.5-mini-instruct/2  (7.64 GB)
  /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/1  (10.48 GB)
  /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2  (10.48 GB)
  /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b/2  (37.00 GB)
  /kaggle/input/models/google/gemma/transformers/2b-it/3  (15.07 GB)
  /kaggle/input/models/google/gemma/transformers/7b-it/3  (51.26 GB)
  /kaggle/input/models/metaresearch/llama-3.1/transformers/8b/2  (32.13 GB)
  /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1  (6.18 GB)
  /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1  (15.24 GB)

CSV/DATASET FILES:
  /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_BEST_BALANCED_1to1.csv  (3.5 MB)

MODEL AVAILABILITY SUMMARY
  [FOUND] Gemma-2-2B
  [FOUND] Qwen2.5-3

In [9]:
# ============================================================
# BENCHMARK SETUP
# ============================================================
import sys

all_results = []
if os.path.exists(RESULTS_FILE):
    try:
        existing = pd.read_csv(RESULTS_FILE)
        all_results = existing.to_dict("records")
        print(f"[{_now()}] Loaded {len(all_results)} existing results", flush=True)
        for r in all_results:
            print(f"  {r['Model']}: Acc={r['Accuracy']}, F1={r['F1']}", flush=True)
    except Exception as e:
        print(f"[{_now()}] Could not load results: {e}", flush=True)

completed = {r["Model"] for r in all_results}
overall_t0 = time.time()

def run_single_model(model_idx):
    """Run one model by index. Resumable — skips completed models."""
    cfg = MODEL_CONFIGS[model_idx]
    model_name = cfg["name"]

    if model_name in completed:
        print(f"[{_now()}] SKIP (already done): {model_name}", flush=True)
        return

    model_path, source = find_model_path(cfg)
    print(f"[{_now()}] Model {model_idx+1}/{len(MODEL_CONFIGS)}: {model_name} [{source}]", flush=True)
    print(f"[{_now()}] Path: {model_path}", flush=True)
    
    # Per-model memory tuning: larger models get smaller seq_len and batch
    global MAX_SEQ_LEN, BATCH_SIZE
    if "9B" in model_name or "8B" in model_name:
        MAX_SEQ_LEN = 256
        BATCH_SIZE = 1
        print(f"[{_now()}] Large model detected — MAX_SEQ_LEN={MAX_SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}", flush=True)
    elif "7B" in model_name:
        MAX_SEQ_LEN = 320
        BATCH_SIZE = 1
        print(f"[{_now()}] 7B model — MAX_SEQ_LEN={MAX_SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}", flush=True)
    else:
        MAX_SEQ_LEN = 384
        BATCH_SIZE = 2
    sys.stdout.flush()

    try:
        metrics = finetune_and_evaluate(model_path, model_name, source)
        all_results.append(metrics)
        completed.add(model_name)
        pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)
        print(f"\n[{_now()}] >>> DONE: {model_name} "
              f"(Acc={metrics['Accuracy']}, F1={metrics['F1']})", flush=True)
    except Exception as e:
        print(f"\n[{_now()}] >>> FAIL: {model_name}: {e}", flush=True)
        import traceback
        traceback.print_exc()
        sys.stdout.flush()
        if all_results:
            pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)

    elapsed = (time.time() - overall_t0) / 60
    print(f"\n[{_now()}] Cumulative: {len(all_results)}/{len(MODEL_CONFIGS)} in {elapsed:.1f} min", flush=True)
    for r in all_results:
        print(f"  {r['Model']:<28} Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}", flush=True)
    sys.stdout.flush()

print(f"[{_now()}] Setup complete. {len(MODEL_CONFIGS)} model cells follow.", flush=True)
print(f"[{_now()}] Completed: {len(completed)}/{len(MODEL_CONFIGS)}", flush=True)

[10:41:30] Setup complete. 1 model cells follow.
[10:41:30] Completed: 0/1


#### Model 1/1: Qwen2.5-3B-Instruct

In [10]:
# Model 1/1: Qwen2.5-3B-Instruct
run_single_model(0)

[10:41:30] Model 1/1: Qwen2.5-3B-Instruct [kaggle]
[10:41:30] Path: /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1

[10:41:30] ============================================================
[10:41:30] Qwen2.5-3B-Instruct
[10:41:30] Source: /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1
[10:41:30] ============================================================
[10:41:30] [Qwen2.5-3B-Instruct] 1/6 Loading tokenizer...
[10:41:31] [Qwen2.5-3B-Instruct] 2/6 Loading model (4-bit NF4, fp16)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[10:41:52] [Qwen2.5-3B-Instruct] VRAM: 2.69 GB
[10:41:52] [Qwen2.5-3B-Instruct] 3/6 Formatting data...


Map:   0%|          | 0/612 [00:00<?, ? examples/s]

[10:41:54] [Qwen2.5-3B-Instruct] 4/6 Setting up LoRA...
[10:41:54] [Qwen2.5-3B-Instruct] LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj']


...

Building labels for train dataset:   0%|          | 0/612 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/612 [00:00<?, ? examples/s]

[10:41:59] [Qwen2.5-3B-Instruct] Trainable: 7,372,800/1,706,045,440 (0.43%)
[10:41:59] [Qwen2.5-3B-Instruct] 5/6 Training 3 epoch(s)...


Step,Training Loss
25,1.265268
50,1.064976
75,1.023814
100,1.008068
125,0.986772
150,0.977135
175,0.969405
200,0.960274
225,0.954702


[10:50:36] [Qwen2.5-3B-Instruct] step 25/231 (10.8%) | epoch=0.33 | loss=1.2653 | lr=1.97e-04
[10:59:14] [Qwen2.5-3B-Instruct] step 50/231 (21.6%) | epoch=0.65 | loss=1.0650 | lr=1.83e-04
[11:07:47] [Qwen2.5-3B-Instruct] step 75/231 (32.5%) | epoch=0.98 | loss=1.0238 | lr=1.59e-04
[11:16:17] [Qwen2.5-3B-Instruct] step 100/231 (43.3%) | epoch=1.30 | loss=1.0081 | lr=1.28e-04
[11:24:56] [Qwen2.5-3B-Instruct] step 125/231 (54.1%) | epoch=1.63 | loss=0.9868 | lr=9.30e-05
[11:33:36] [Qwen2.5-3B-Instruct] step 150/231 (64.9%) | epoch=1.95 | loss=0.9771 | lr=5.92e-05
[11:42:08] [Qwen2.5-3B-Instruct] step 175/231 (75.8%) | epoch=2.27 | loss=0.9694 | lr=3.03e-05
[11:50:46] [Qwen2.5-3B-Instruct] step 200/231 (86.6%) | epoch=2.60 | loss=0.9603 | lr=9.90e-06
[11:59:19] [Qwen2.5-3B-Instruct] step 225/231 (97.4%) | epoch=2.93 | loss=0.9547 | lr=4.82e-07
[12:01:14] [Qwen2.5-3B-Instruct] step 231/231 (100.0%) | epoch=3.00 | loss=0.0000 | lr=0.00e+00
[12:01:14] [Qwen2.5-3B-Instruct] Training: 79.2 min


### 9.11 Final Summary

After all models have been evaluated, a final summary table is printed and saved alongside the per-model results.

In [11]:
# Final summary
total_min = (time.time() - overall_t0) / 60
print(f"\n{'='*60}", flush=True)
print(f"  ALL DONE in {total_min:.1f} min ({total_min/10:.1f} hours)", flush=True)
print(f"  Completed: {len(all_results)}/{len(MODEL_CONFIGS)}", flush=True)
print(f"  Results: {RESULTS_FILE}", flush=True)
print(f"{'='*60}", flush=True)
print(f"\nFinal results:", flush=True)
for r in sorted(all_results, key=lambda x: -x.get('F1', 0)):
    print(f"  {r['Model']:<28} Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}  "
          f"Kappa={r['Kappa']:.4f}  Src={r.get('Source','?')}", flush=True)
sys.stdout.flush()


  ALL DONE in 81.1 min (8.1 hours)
  Completed: 1/1
  Results: /kaggle/working/local_llm_results.csv

Final results:
  Qwen2.5-3B-Instruct          Acc=0.5065  F1=0.0500  Kappa=0.0130  Src=kaggle


### 10. Comprehensive Comparison Table

The fine-tuned LLM results are aggregated alongside the BanglaBERT Large baseline (5-fold cross-validation on the same 766-sample clean balanced dataset) for direct comparison.

In [12]:
results_df = pd.DataFrame(all_results)
refs = [{"Model": "BanglaBERT Large (5-fold CV)", "Accuracy": 0.8652, "Precision": 0.8671,
         "Recall": 0.8527, "F1": 0.8527, "Kappa": 0.7304, "MCC": 0.7305,
         "Source": "fine-tuned"}]
comp = pd.concat([pd.DataFrame(refs), results_df], ignore_index=True) if len(results_df) > 0 else pd.DataFrame(refs)
dcols = ["Model", "Accuracy", "F1", "Kappa", "MCC", "Source"]
print("\n" + "="*80, flush=True)
print("  COMPREHENSIVE COMPARISON — Local LLM Fine-Tuning vs BanglaBERT", flush=True)
print("="*80, flush=True)
print(comp[dcols].to_string(index=False), flush=True)
comp.to_csv(f"{OUTPUT_DIR}/full_comparison_table.csv", index=False)
print(f"\n[{_now()}] Saved: {OUTPUT_DIR}/full_comparison_table.csv", flush=True)


  COMPREHENSIVE COMPARISON — Local LLM Fine-Tuning vs BanglaBERT
                       Model  Accuracy     F1  Kappa    MCC     Source
BanglaBERT Large (5-fold CV)    0.8652 0.8527 0.7304 0.7305 fine-tuned
         Qwen2.5-3B-Instruct    0.5065 0.0500 0.0130 0.0470     kaggle

[12:02:36] Saved: /kaggle/working/full_comparison_table.csv


### 11. Visualisation

In [13]:
if len(results_df) > 0:
    n = len(results_df)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (mname, ref) in zip(axes, [("Accuracy",0.8652),("F1",0.8527),("Kappa",0.7304)]):
        vals = results_df[mname].values
        names = results_df["Model"].values
        short = [s.replace("-Instruct","").replace("-v0.","-v") for s in names]
        bars = ax.barh(range(n), vals, color=plt.cm.tab10(np.linspace(0,1,n)))
        ax.axvline(ref, color="red", ls="--", alpha=0.7, label=f"BanglaBERT ({ref})")
        ax.set_yticks(range(n))
        ax.set_yticklabels(short, fontsize=8)
        ax.set_xlabel(mname)
        ax.set_title(mname, fontsize=12)
        ax.legend(fontsize=7)
        for bar, v in zip(bars, vals):
            ax.text(v+0.002, bar.get_y()+bar.get_height()/2, f"{v:.4f}", va="center", fontsize=7)
        ax.invert_yaxis()
    plt.suptitle("Local LLM Fine-Tuning vs BanglaBERT", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[{_now()}] Saved: model_comparison.png", flush=True)
else:
    print("No results to visualize.", flush=True)

[12:02:36] Saved: model_comparison.png


### 12. Final Results Export

In [14]:
results_df.to_csv(RESULTS_FILE, index=False)
summary = {
    "dataset": "Swarabyanjan_BEST_BALANCED_1to1",
    "train_size": int(len(train_df)), "test_size": int(len(test_df)),
    "seed": SEED, "n_models_completed": len(all_results),
    "results": all_results,
}
with open(f"{OUTPUT_DIR}/results_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
print(f"[{_now()}] Results: {RESULTS_FILE}", flush=True)
print(f"[{_now()}] Summary: {OUTPUT_DIR}/results_summary.json", flush=True)
print(f"\n[{_now()}] Output files:", flush=True)
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:<45} {f.stat().st_size/1024:>8.1f} KB", flush=True)

[12:02:36] Results: /kaggle/working/local_llm_results.csv
[12:02:36] Summary: /kaggle/working/results_summary.json

[12:02:36] Output files:
  __notebook__.ipynb                               144.1 KB
  cm_Qwen2_5-3B-Instruct.png                        22.8 KB
  full_comparison_table.csv                          0.3 KB
  local_llm_results.csv                              0.2 KB
  model_comparison.png                              49.3 KB
  results_summary.json                               0.5 KB
